In [0]:
# import everyting
from pyspark.pandas import DataFrame
from pyspark.sql import SparkSession
from pyspark.sql.functions import regexp_replace, col, udf, initcap
from dateutil.parser import parser
from datetime import datetime
import re

In [0]:
spark = SparkSession.builder.getOrCreate()

In [0]:
base_path = "s3://bucket-retail-sales/raw_data/"

df_crm_cust_base = spark.read.format("csv").option("header", "true").load(base_path + "CRM_CUST_BASE_RAW.csv")
df_inv_levels    = spark.read.format("csv").option("header", "true").load(base_path + "INV_LEVELS_RAW.csv")
df_product       = spark.read.format("csv").option("header", "true").load(base_path + "MST_PRODUCT_MASTER.csv")
df_sales         = spark.read.format("csv").option("header", "true").load(base_path + "RAW_SALES_DTL.csv")
df_stores_geo    = spark.read.format("csv").option("header", "true").load(base_path + "STORES_GEO_LIST_FINAL.csv")
df_rtn_trans     = spark.read.format("csv").option("header", "true").load(base_path + "RTN_TRANS_LOGS.csv")

def clean_column_names(df):
    def to_snake_case(name):
        cleaned = re.sub(r'[^a-zA-Z0-9]', '', name.strip())
        s1 = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', cleaned)
        snake = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', s1)
        return re.sub(r'_+', '_', snake).lower()
    return df.toDF(*[to_snake_case(c) for c in df.columns])

df_crm_cust_base = clean_column_names(df_crm_cust_base)
df_inv_levels    = clean_column_names(df_inv_levels)
df_product       = clean_column_names(df_product)
df_sales         = clean_column_names(df_sales)
df_stores_geo    = clean_column_names(df_stores_geo)
df_rtn_trans     = clean_column_names(df_rtn_trans)

# Standardise join keys
df_sales     = df_sales.withColumnRenamed("cust_id99", "cust_ref_id") \
                       .withColumnRenamed("prodcodeid", "product_id") \
                       .withColumnRenamed("store_loc_id", "store_id")
df_inv_levels = df_inv_levels.withColumnRenamed("stid_ref", "store_id")
df_rtn_trans  = df_rtn_trans.withColumnRenamed("rtn_date_string", "rtn_date")

In [0]:
# Display Everything
display(df_crm_cust_base)
display(df_inv_levels)
display(df_product)
display(df_sales)
display(df_stores_geo)
display(df_rtn_trans)

In [0]:
# Data cleaning functions
@udf
def fix_date(date_str):
    return str(parser().parse(date_str).strftime("%Y-%m-%d"))

def fix_name(name):
    s1 = regexp_replace(name, "[\"\'\(\)\.]", "")
    return initcap(regexp_replace(s1, "[_]", " "))

@udf
def fix_phone(phone):
    phone = phone.strip().lower()
    if phone.isalpha() or phone == "null":
        return 'UNKNOWN'

    if "e+" in phone:
        return "{:.0f}".format(float(phone))
    phone = re.sub('[ \-+]', '', phone)
    if len(phone) < 12:
        phone = "91" + phone

    return phone

def fix_price(column):
    return regexp_replace(col(column), r"[^\d.]", "").cast("double")

@udf
def fix_c_id(_id: str):
    if _id.startswith('C-'):
        return _id
    return 'UNKNOWN'


In [0]:
df_crm_cust_base = df_crm_cust_base\
.withColumn("full_name", fix_name(col("full_name")))\
.withColumn("joined_date", fix_date(col("joined_date")))\
.withColumn("contact_info", fix_phone(col("contact_info")))

df_inv_levels = df_inv_levels\
.withColumn("last_audit_dt", fix_date(col("last_audit_dt")))

df_sales = df_sales\
.withColumn("date_ref", fix_date(col("date_ref")))\
.withColumn("unit_price", fix_price(col("unit_price")))\
.withColumn("cust_ref_id", fix_c_id(col("cust_ref_id")))

df_rtn_trans = df_rtn_trans\
.withColumn("rtn_date", fix_date(col("rtn_date")))

In [0]:
# Display Everything Bro
display(df_crm_cust_base)
display(df_inv_levels)
display(df_product)
display(df_sales)
display(df_stores_geo)
display(df_rtn_trans)

In [0]:
tables = {
    "crm_cust_base": df_crm_cust_base,
    "inv_levels": df_inv_levels,
    "products": df_product,
    "sales": df_sales,
    "stores_geo": df_stores_geo,
    "rtn_trans": df_rtn_trans
}

for name, df in tables.items():
    df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"workspace.silver.{name}")